# CS-781 Capstone — Phase 5: Legal Statute Identification (LSI)

Multi-label statute prediction using the four Phase-1 Legal Language Models.
This notebook is designed for a Kaggle T4 GPU and addresses the `c10::Half` overflow bugs from earlier iterations.

**Experiments Included:**
1. Model A + LoRMA (compulsory)
2. Model B + Full Fine-tuning
3. Model C + AdaLoRA
4. Model D + Adapters
5. Model A $\to$ 6-layer Model A Knowledge Distillation (compulsory)
6. Probability-level Ensemble + Recalibrated Thresholds

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import sys, json, math, time, random, zipfile, gc, glob, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(42)

class CFG:
    TOK_GLOB = "/kaggle/input/**/bpe32k.json"
    CKPT_GLOBS = {
        "A": "/kaggle/input/**/A_decoder.pt",
        "B": "/kaggle/input/**/B_encdec.pt",
        "C": "/kaggle/input/**/C_hybrid.pt",
        "D": "/kaggle/input/**/D_gqa.pt",
    }
    DIM, N_HEADS, MAX_SEQ = 768, 12, 2048

    # Long documents: sample beginning, middle and ending regions.
    CHUNK_LEN = 512
    MAX_CHUNKS = 3
    CHUNK_POOL = "attn"
    BIDIRECTIONAL = True

    # Runtime controls
    BATCH_SIZE = 2
    GRAD_ACCUM = 4
    EPOCHS = 2
    MAX_TRAIN_DOCS = None       # Set to an integer (e.g., 6000) if 12h Kaggle timeout is an issue
    MAX_STEPS_PER_EXPERIMENT = 2000
    LR_BACKBONE = 2.0e-5
    LR_HEAD = 1.0e-3
    LR_PEFT = 1.0e-3
    WARMUP_FRAC = 0.05
    WEIGHT_DECAY = 0.01
    MAX_GRAD_NORM = 1.0
    POS_WEIGHT_CAP = 10.0
    AMP = True

    LORMA_R, LORMA_ALPHA = 16, 32
    ADALORA_R, ADALORA_ALPHA = 16, 32
    ADALORA_TARGET_R = 8
    ADALORA_ORTH = 0.05
    ADAPTER_DIM = 64

    KD_T, KD_ALPHA = 2.0, 0.5
    OUT_DIR = "/kaggle/working/phase5_submissions"
    DATA_DIR = "/kaggle/working/p5"

cfg = CFG()
os.makedirs(cfg.OUT_DIR, exist_ok=True)
os.makedirs(cfg.DATA_DIR, exist_ok=True)

def find_one(pattern, what):
    hits = sorted(glob.glob(pattern, recursive=True))
    assert hits, f"{what} not found: {pattern}. Attach the relevant Kaggle dataset."
    return Path(hits[0])

TOK_PATH = find_one(cfg.TOK_GLOB, "bpe32k.json")
CKPTS = {k: find_one(v, f"Phase-1 model {k}") for k,v in cfg.CKPT_GLOBS.items()}
print("Tokenizer:", TOK_PATH)
for k,v in CKPTS.items(): print(f"Model {k}: {v} ({v.stat().st_size/1e6:.1f} MB)")

# Download Phase 5 from the HF dataset repository using the Kaggle Secret.
p5_train_hits = glob.glob("/kaggle/**/Phase_5_LSI/**/train.json", recursive=True)
if not p5_train_hits:
    from huggingface_hub import snapshot_download
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        raise RuntimeError("Add HF_TOKEN under Kaggle Secrets before running.") from e
    snapshot_download(
        repo_id="Exploration-Lab/CS-781-Capstone", repo_type="dataset",
        allow_patterns=["Phase_5_LSI/**"], local_dir=cfg.DATA_DIR, token=hf_token
    )
else:
    print("Phase 5 data already attached:", p5_train_hits[0])

## 2. Load LSI data and build the statute-ID vocabulary

In [ ]:
def read_json(path):
    with open(path, "r", encoding="utf-8") as f: return json.load(f)

def locate_p5(name):
    patterns = [
        f"/kaggle/**/Phase_5_LSI/**/{name}",
        f"{cfg.DATA_DIR}/**/{name}",
    ]
    hits=[]
    for pat in patterns: hits += glob.glob(pat, recursive=True)
    assert hits, f"Cannot find Phase_5_LSI/{name}"
    return sorted(hits)[0]

def load_lsi(path, with_labels=True):
    raw = read_json(path)
    if isinstance(raw, dict):
        items = [(v if isinstance(v,dict) else {"text":v}) | {"id":k} for k,v in raw.items()]
    elif isinstance(raw, list):
        items = raw
    else:
        raise ValueError(f"Unsupported JSON structure in {path}")
    out={}
    for i,d in enumerate(items):
        did=str(d.get("id", d.get("case_id", d.get("doc_id", i))))
        txt=d.get("text", d.get("document", d.get("case_text", "")))
        if isinstance(txt, list): txt=" ".join(map(str,txt))
        rec={"text": str(txt or "")}
        if with_labels:
            lab=d.get("labels", d.get("label", d.get("statutes", [])))
            if lab is None: lab=[]
            if isinstance(lab, (int, str)): lab=[lab]
            rec["labels"]=sorted(set(int(x) for x in lab))
        out[did]=rec
    return out

TRAIN_PATH, DEV_PATH, TEST_PATH = [locate_p5(x) for x in ("train.json","dev.json","test.json")]
train_docs=load_lsi(TRAIN_PATH, True)
dev_docs=load_lsi(DEV_PATH, True)
test_docs=load_lsi(TEST_PATH, False)

STATUTE_IDS=sorted({s for d in train_docs.values() for s in d["labels"]})
assert STATUTE_IDS, "No statute labels found in train.json"
SID2COL={s:i for i,s in enumerate(STATUTE_IDS)}
N_LABELS=len(STATUTE_IDS)

for split,docs in [("train",train_docs),("dev",dev_docs),("test",test_docs)]:
    print(f"{split}: {len(docs):,} documents")
print("Statute classes:", N_LABELS, "| ID range:", (min(STATUTE_IDS),max(STATUTE_IDS)))
freq=Counter(s for d in train_docs.values() for s in d["labels"])
print("Labels/document train: mean", round(np.mean([len(d["labels"]) for d in train_docs.values()]),2),
      "| median", int(np.median([len(d["labels"]) for d in train_docs.values()])))
assert all(set(d["labels"]).issubset(SID2COL) for d in dev_docs.values()), "Dev contains unseen statute IDs" 

## 3. Tokenizer, long-document sampling and batches

In [ ]:
from tokenizers import Tokenizer
tok=Tokenizer.from_file(str(TOK_PATH))
VOCAB_SIZE=tok.get_vocab_size()
pad_token_id=tok.token_to_id("<|pad|>")
PAD_ID=int(pad_token_id if pad_token_id is not None else 0)
print("Tokenizer vocab:",VOCAB_SIZE,"| pad:",PAD_ID)

_ENC_CACHE={}
def doc_chunks(did,text):
    key=(did,cfg.CHUNK_LEN,cfg.MAX_CHUNKS)
    if key not in _ENC_CACHE:
        ids=tok.encode(text).ids or [PAD_ID]
        L=cfg.CHUNK_LEN
        chunks=[ids[i:i+L] for i in range(0,len(ids),L)]
        if len(chunks)>cfg.MAX_CHUNKS:
            # Preserve beginning, middle and ending context
            ix=np.linspace(0,len(chunks)-1,cfg.MAX_CHUNKS).round().astype(int)
            chunks=[chunks[j] for j in sorted(set(ix))]
        _ENC_CACHE[key]=chunks or [[PAD_ID]]
    return _ENC_CACHE[key]

class LSIDocs(Dataset):
    def __init__(self,docs,with_labels=True):
        self.ids=list(docs); self.docs=docs; self.with_labels=with_labels
    def __len__(self): return len(self.ids)
    def __getitem__(self,i):
        did=self.ids[i]; d=self.docs[did]
        y=np.zeros(N_LABELS,dtype=np.float32)
        if self.with_labels:
            for s in d["labels"]:
                if s in SID2COL: y[SID2COL[s]]=1.
        return {"did":did,"chunks":doc_chunks(did,d["text"]),"y":y}

def collate(batch):
    flat=[]; owner=[]
    for bi,b in enumerate(batch):
        for ch in b["chunks"]:
            flat.append(ch); owner.append(bi)
    T=max(len(x) for x in flat)
    ids=torch.full((len(flat),T),PAD_ID,dtype=torch.long)
    mask=torch.zeros((len(flat),T),dtype=torch.bool)
    for i,ch in enumerate(flat):
        ids[i,:len(ch)]=torch.tensor(ch,dtype=torch.long)
        mask[i,:len(ch)]=True
    return {"input_ids":ids,"tok_mask":mask,
            "owner":torch.tensor(owner,dtype=torch.long),"n_docs":len(batch),
            "labels":torch.tensor(np.stack([b["y"] for b in batch]),dtype=torch.float32),
            "dids":[b["did"] for b in batch]}

def select_training_docs(docs, cap):
    if cap is None or len(docs)<=cap: return docs
    rng=np.random.default_rng(42)
    ids=list(docs)
    f=Counter(s for d in docs.values() for s in d["labels"])
    order=sorted(ids,key=lambda did:min(f[s] for s in docs[did]["labels"]))
    chosen=[]; seen=set()
    for did in order:
        if len(chosen)>=cap: break
        if rng.random()<0.65 or len(chosen)<min(cap//2,len(order)//3):
            chosen.append(did); seen.add(did)
    if len(chosen)<cap:
        rest=[x for x in ids if x not in seen]
        rng.shuffle(rest); chosen.extend(rest[:cap-len(chosen)])
    return {k:docs[k] for k in chosen[:cap]}

train_docs_used=select_training_docs(train_docs,cfg.MAX_TRAIN_DOCS)
if len(train_docs_used)<len(train_docs):
    print(f"Training subset: {len(train_docs_used):,}/{len(train_docs):,}; change CFG.MAX_TRAIN_DOCS to use more.")
ds_train=LSIDocs(train_docs_used,True)
ds_dev=LSIDocs(dev_docs,True)
ds_test=LSIDocs(test_docs,False)

## 4. Phase-1 architecture primitives

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        dtype = x.dtype
        x = x.float()
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x * rms).to(dtype) * self.weight

class SwiGLU(nn.Module):
    def __init__(self, dim, mult=4, multiple_of=64):
        super().__init__()
        hidden = int(dim * mult * 2 / 3)
        hidden = multiple_of * ((hidden + multiple_of - 1) // multiple_of)
        self.w1 = nn.Linear(dim, hidden, bias=False)
        self.w2 = nn.Linear(dim, hidden, bias=False)
        self.w3 = nn.Linear(hidden, dim, bias=False)
    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))

class RoPE(nn.Module):
    def __init__(self, head_dim, max_seq):
        super().__init__()
        self.head_dim, self.max_seq = head_dim, max_seq
        inv = 1.0 / (10000 ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        fr = torch.outer(torch.arange(max_seq, dtype=torch.float32), inv)
        self.register_buffer("cos", fr.cos(), persistent=False)
        self.register_buffer("sin", fr.sin(), persistent=False)
    def forward(self, x, pos0=0):
        dtype = x.dtype
        B, H, T, D = x.shape
        half = D // 2
        x = x.float()
        x1, x2 = x[..., :half], x[..., half:]
        cos = self.cos[pos0:pos0 + T].unsqueeze(0).unsqueeze(0)
        sin = self.sin[pos0:pos0 + T].unsqueeze(0).unsqueeze(0)
        return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], -1).to(dtype)

class CausalSelfAttention(nn.Module):
    def __init__(self, dim, n_heads, max_seq, rope=True):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, dim // n_heads
        self.qkv      = nn.Linear(dim, 3 * dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.rope     = RoPE(self.head_dim, max_seq) if rope else None
        self.causal   = True
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        if self.rope is not None:
            q, k = self.rope(q), self.rope(k)
        o = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal)
        return self.out_proj(o.transpose(1, 2).reshape(B, T, C))

class GeneralizedAttention(nn.Module):
    def __init__(self, dim, n_heads, max_seq, causal=True, cross=False):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, dim // n_heads
        self.causal, self.cross = causal, cross
        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.rope = None if cross else RoPE(self.head_dim, max_seq)
    def forward(self, x, context=None):
        B, T, C = x.shape
        src = context if self.cross else x
        S = src.shape[1]
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(src).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(src).view(B, S, self.n_heads, self.head_dim).transpose(1, 2)
        if self.rope is not None:
            q, k = self.rope(q), self.rope(k)
        o = F.scaled_dot_product_attention(q, k, v, is_causal=(self.causal and not self.cross))
        return self.out_proj(o.transpose(1, 2).reshape(B, T, C))

class EncoderBlock(nn.Module):
    def __init__(self, dim, n_heads, max_seq):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn  = GeneralizedAttention(dim, n_heads, max_seq, causal=False, cross=False)
        self.norm2 = RMSNorm(dim)
        self.mlp   = SwiGLU(dim)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))

class DecoderBlock(nn.Module):
    def __init__(self, dim, n_heads, max_seq):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.self_attn = GeneralizedAttention(dim, n_heads, max_seq, causal=True, cross=False)
        self.norm2 = RMSNorm(dim)
        self.cross_attn = GeneralizedAttention(dim, n_heads, max_seq, causal=False, cross=True)
        self.norm3 = RMSNorm(dim)
        self.mlp = SwiGLU(dim)
    def forward(self, x, enc):
        x = x + self.self_attn(self.norm1(x))
        x = x + self.cross_attn(self.norm2(x), context=enc)
        return x + self.mlp(self.norm3(x))

class GatedLinearRecurrence(nn.Module):
    def __init__(self, dim, conv_kernel=3, chunk_size=16):
        super().__init__()
        self.dim, self.chunk_size, self.conv_kernel = dim, chunk_size, conv_kernel
        self.in_proj = nn.Linear(dim, 4 * dim, bias=False)
        self.dwconv = nn.Conv1d(dim, dim, conv_kernel, padding=conv_kernel - 1, groups=dim, bias=False)
        self.decay_proj = nn.Linear(dim, dim, bias=True)
        self.out_proj   = nn.Linear(dim, dim, bias=False)
    def _scan(self, log_a, u):
        B, T, D = u.shape
        L = self.chunk_size
        pad = (-T) % L
        if pad:
            log_a = F.pad(log_a, (0, 0, 0, pad)); u = F.pad(u, (0, 0, 0, pad))
        n = (T + pad) // L
        log_a = log_a.view(B, n, L, D); u = u.view(B, n, L, D)
        A = log_a.cumsum(2)
        d = A.unsqueeze(3) - A.unsqueeze(2)
        tri = torch.tril(torch.ones(L, L, device=u.device, dtype=torch.bool))
        d = d.masked_fill(~tri[None, None, :, :, None], float("-inf"))
        local = torch.einsum("bnijd,bnjd->bnid", torch.exp(d), u)
        h, outs = u.new_zeros(B, D), []
        decay = torch.exp(A)
        for c in range(n):
            outs.append(local[:, c] + decay[:, c] * h[:, None])
            h = outs[-1][:, -1]
        return torch.cat(outs, 1)[:, :T]
    def forward(self, x):
        B, T, D = x.shape
        c = self.dwconv(x.transpose(1, 2))[..., :T].transpose(1, 2)
        v, i, _, o = self.in_proj(c).chunk(4, dim=-1)
        v, i, o = torch.tanh(v), torch.sigmoid(i), torch.sigmoid(o)
        log_a = -0.01 - 0.99 * torch.sigmoid(self.decay_proj(c).float())
        h = self._scan(log_a, i.float() * v.float())
        return self.out_proj((h * o.float()).to(x.dtype))

class GQASlidingWindowAttention(nn.Module):
    def __init__(self, dim, n_heads, n_kv_heads=4, max_seq=2048, window_size=256, sliding_window=True):
        super().__init__()
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.head_dim = dim // n_heads
        self.window_size, self.sliding_window = window_size, sliding_window
        self.q_proj = nn.Linear(dim, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.rope = RoPE(self.head_dim, max_seq)
        pos = torch.arange(max_seq)
        d = pos[:, None] - pos[None, :]
        causal = d >= 0
        self.register_buffer("causal_mask", causal, persistent=False)
        self.register_buffer("window_mask", causal & (d < window_size), persistent=False)
        self.register_buffer("bi_full_mask", torch.ones_like(causal), persistent=False)
        self.register_buffer("bi_window_mask", d.abs() < window_size, persistent=False)
        self.bidirectional = False
    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads,    self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q, k = self.rope(q), self.rope(k)
        if self.bidirectional:
            m = self.bi_window_mask if self.sliding_window else self.bi_full_mask
        else:
            m = self.window_mask if self.sliding_window else self.causal_mask
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=m[:T, :T], is_causal=False, enable_gqa=(self.n_kv_heads != self.n_heads))
        return self.out_proj(o.transpose(1, 2).reshape(B, T, C))

class GenericBlock(nn.Module):
    def __init__(self, dim, mixer):
        super().__init__()
        self.norm1, self.mixer = RMSNorm(dim), mixer
        self.norm2, self.mlp   = RMSNorm(dim), SwiGLU(dim)
    def forward(self, x):
        x = x + self.mixer(self.norm1(x))
        return x + self.mlp(self.norm2(x))

## 5. Phase-1 backbone builders

In [ ]:
DIM, N_HEADS, MAX_SEQ = cfg.DIM, cfg.N_HEADS, cfg.MAX_SEQ

class TransformerBlock(nn.Module):
    def __init__(self, dim, n_heads, max_seq):
        super().__init__()
        self.norm1 = RMSNorm(dim); self.attn = CausalSelfAttention(dim, n_heads, max_seq)
        self.norm2 = RMSNorm(dim); self.mlp  = SwiGLU(dim)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))

class GPTModel(nn.Module):
    def __init__(self, vocab_size, dim=DIM, n_layers=12, n_heads=N_HEADS, max_seq=MAX_SEQ):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.blocks  = nn.ModuleList([TransformerBlock(dim, n_heads, max_seq) for _ in range(n_layers)])
        self.norm_f  = RMSNorm(dim)
        self.head    = nn.Linear(dim, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
    def trunk_embeds(self, x):
        for b in self.blocks: x = b(x)
        return self.norm_f(x)
    def trunk(self, idx): return self.trunk_embeds(self.tok_emb(idx))

class EncoderDecoderModel(nn.Module):
    def __init__(self, vocab_size, dim=DIM, n_encoder_layers=6, n_decoder_layers=6, n_heads=N_HEADS, max_seq=MAX_SEQ):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.encoder_blocks = nn.ModuleList([EncoderBlock(dim, n_heads, max_seq) for _ in range(n_encoder_layers)])
        self.norm_e = RMSNorm(dim)
        self.decoder_blocks = nn.ModuleList([DecoderBlock(dim, n_heads, max_seq) for _ in range(n_decoder_layers)])
        self.norm_f = RMSNorm(dim)
        self.head = nn.Linear(dim, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
    def trunk_embeds(self, x):
        e = x
        for b in self.encoder_blocks: e = b(e)
        e = self.norm_e(e)
        d = x
        for b in self.decoder_blocks: d = b(d, e)
        return self.norm_f(d)
    def trunk(self, idx): return self.trunk_embeds(self.tok_emb(idx))

class GenericLM(nn.Module):
    def __init__(self, mixer_factory, vocab_size, dim=DIM, n_layers=12, max_seq=MAX_SEQ):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.blocks  = nn.ModuleList([GenericBlock(dim, mixer_factory(i)) for i in range(n_layers)])
        self.norm_f  = RMSNorm(dim)
        self.head    = nn.Linear(dim, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
    def trunk_embeds(self, x):
        for b in self.blocks: x = b(x)
        return self.norm_f(x)
    def trunk(self, idx): return self.trunk_embeds(self.tok_emb(idx))

C_LAYERS, C_ATTN_EVERY, C_CHUNK = 12, 3, 16
D_LAYERS, D_FULL_EVERY, D_WINDOW, D_KV_HEADS = 12, 3, 256, 4

def c_mixer_factory(i):
    if (i + 1) % C_ATTN_EVERY == 0: return CausalSelfAttention(DIM, N_HEADS, max_seq=MAX_SEQ)
    return GatedLinearRecurrence(DIM, conv_kernel=3, chunk_size=C_CHUNK)

def d_mixer_factory(i):
    return GQASlidingWindowAttention(DIM, N_HEADS, n_kv_heads=D_KV_HEADS, max_seq=MAX_SEQ,
                                     window_size=D_WINDOW, sliding_window=((i + 1) % D_FULL_EVERY != 0))

def _build(arch):
    if arch == "A": return GPTModel(VOCAB_SIZE, n_layers=12)
    if arch == "B": return EncoderDecoderModel(VOCAB_SIZE)
    if arch == "C": return GenericLM(c_mixer_factory, VOCAB_SIZE, n_layers=C_LAYERS)
    if arch == "D": return GenericLM(d_mixer_factory, VOCAB_SIZE, n_layers=D_LAYERS)
    raise ValueError(arch)

def trunk_blocks(m, arch): return m.decoder_blocks if arch == "B" else m.blocks

def load_phase1(arch, pretrained=True):
    m = _build(arch)
    if pretrained:
        sd = torch.load(CKPTS[arch], map_location="cpu")
        if isinstance(sd, dict) and "model" in sd: sd = sd["model"]
        m.load_state_dict({k: (v.float() if v.is_floating_point() else v) for k, v in sd.items()}, strict=True)
    return m

ARCH_INFO = {"A": "decoder-only, 12L", "B": "encoder-decoder, 6+6L (full stack)",
             "C": "GLR + attention hybrid, 12L", "D": "GQA sliding-window, 12L"}

## 6. Multi-label document classifier

In [ ]:
def set_bidirectional(model,on=True):
    n=0
    for mod in model.modules():
        if isinstance(mod,CausalSelfAttention): mod.causal=not on; n+=1
        elif isinstance(mod,GQASlidingWindowAttention): mod.bidirectional=on; n+=1
        elif isinstance(mod,GeneralizedAttention) and not mod.cross and mod.causal: mod.causal=not on; n+=1
    return n

class LSIModel(nn.Module):
    def __init__(self,backbone,arch,hidden,dropout=0.15):
        super().__init__()
        self.backbone,self.arch=backbone,arch
        self.chunk_attn=nn.Sequential(nn.Linear(hidden,hidden//4),nn.Tanh(),nn.Linear(hidden//4,1))
        self.drop=nn.Dropout(dropout)
        self.head=nn.Sequential(nn.Linear(hidden,hidden),nn.GELU(),nn.Dropout(dropout),
                                nn.Linear(hidden,N_LABELS))
    def forward(self,input_ids,tok_mask,owner,n_docs):
        h=self.backbone.trunk(input_ids)
        m=tok_mask.unsqueeze(-1).to(h.dtype)
        cvec=(h*m).sum(1)/m.sum(1).clamp(min=1e-6)
        counts=torch.bincount(owner,minlength=n_docs)
        cmax=int(counts.max().item())
        slot=torch.cat([torch.arange(int(c),device=owner.device) for c in counts]) if cmax>1 else torch.zeros_like(owner)
        P=cvec.new_zeros(n_docs,cmax,cvec.size(-1))
        P[owner,slot]=cvec
        valid=torch.zeros(n_docs,cmax,dtype=torch.bool,device=cvec.device)
        valid[owner,slot]=True
        if cmax==1: doc=P[:,0]
        else:
            # Fix AMP overflow: use a safe small value instead of torch.finfo(dtype).min
            scores=self.chunk_attn(P).squeeze(-1).masked_fill(~valid, -1e4)
            doc=(P*scores.softmax(-1).unsqueeze(-1)).sum(1)
        return self.head(self.drop(doc))

def build_lsi(arch,keep_layers=None):
    bb=load_phase1(arch,pretrained=True)
    if keep_layers is not None:
        blocks=trunk_blocks(bb,arch)
        keep=np.linspace(0,len(blocks)-1,keep_layers).round().astype(int).tolist()
        new=nn.ModuleList([blocks[i] for i in keep])
        if arch=="B": bb.decoder_blocks=new
        else: bb.blocks=new
    bb.head=nn.Identity()
    n=set_bidirectional(bb,cfg.BIDIRECTIONAL)
    model=LSIModel(bb,arch,cfg.DIM).to(DEVICE)
    return model

## 7. Adaptation methods: LoRMA, AdaLoRA and Adapters

In [ ]:
class LoRMALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        for p in self.base.parameters(): p.requires_grad = False
        d = base.out_features
        self.A = nn.Parameter(torch.zeros(r, d))
        self.B = nn.Parameter(torch.zeros(d, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / r
    def forward(self, x):
        y = self.base(x)
        return y + ((y @ self.A.t()) @ self.B.t()) * self.scale

class AdaLoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        for p in self.base.parameters(): p.requires_grad = False
        self.Q = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        self.P = nn.Parameter(torch.randn(base.out_features, r) * 0.02)
        self.lam = nn.Parameter(torch.zeros(r))
        self.register_buffer("mask", torch.ones(r))
        self.register_buffer("imp", torch.zeros(r))
        self.scale, self.r = alpha / r, r
    def forward(self, x):
        z = (x @ self.Q.t()) * (self.lam * self.mask)
        return self.base(x) + (z @ self.P.t()) * self.scale
    def orth_penalty(self):
        I = torch.eye(self.r, device=self.P.device, dtype=self.P.dtype)
        return ((self.P.t() @ self.P - I).pow(2).mean() + (self.Q @ self.Q.t() - I).pow(2).mean())
    @torch.no_grad()
    def update_importance(self, beta=0.85):
        if self.lam.grad is not None:
            self.imp.mul_(beta).add_((1 - beta) * (self.lam * self.lam.grad).abs())

def adalora_housekeeping(model, step, total):
    mods = [m for m in model.modules() if isinstance(m, AdaLoRALinear)]
    for m in mods: m.update_importance()
    if mods and step == max(1, total // 2):
        allimp = torch.cat([m.imp for m in mods])
        budget = max(1, int(cfg.ADALORA_TARGET_R * len(mods)))
        if budget < allimp.numel():
            threshold = torch.topk(allimp, budget).values.min()
            for m in mods: m.mask.copy_((m.imp >= threshold).float())

class AdapterWrap(nn.Module):
    def __init__(self, base, bottleneck):
        super().__init__()
        self.base = base
        for p in self.base.parameters(): p.requires_grad = False
        d = base.out_features
        self.down = nn.Linear(d, bottleneck)
        self.up   = nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, std=1e-2); nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight);              nn.init.zeros_(self.up.bias)
    def forward(self, x):
        y = self.base(x)
        return y + self.up(F.gelu(self.down(y)))

MIXERS = (CausalSelfAttention, GeneralizedAttention, GQASlidingWindowAttention, GatedLinearRecurrence)
def mixer_linears(model):
    for mod in model.modules():
        if isinstance(mod, MIXERS):
            for cname, child in list(mod.named_children()):
                if isinstance(child, nn.Linear): yield mod, cname, child

def apply_adaptation(model,method):
    bb,arch=model.backbone,model.arch
    for p in bb.parameters(): p.requires_grad=False
    for p in model.head.parameters(): p.requires_grad=True
    for p in model.chunk_attn.parameters(): p.requires_grad=True
    blocks=trunk_blocks(bb,arch)
    if method=="full":
        for p in bb.parameters(): p.requires_grad=True
    elif method=="lorma":
        for parent,cn,lin in list(mixer_linears(bb)):
            setattr(parent,cn,LoRMALinear(lin,cfg.LORMA_R,cfg.LORMA_ALPHA).to(DEVICE))
    elif method=="adalora":
        for parent,cn,lin in list(mixer_linears(bb)):
            setattr(parent,cn,AdaLoRALinear(lin,cfg.ADALORA_R,cfg.ADALORA_ALPHA).to(DEVICE))
    elif method=="adapter":
        for parent,cn,lin in list(mixer_linears(bb)):
            if cn=="out_proj": setattr(parent,cn,AdapterWrap(lin,cfg.ADAPTER_DIM).to(DEVICE))
        for blk in blocks:
            mlp=getattr(blk,"mlp",None)
            if mlp is not None and isinstance(mlp.w3,nn.Linear):
                mlp.w3=AdapterWrap(mlp.w3,cfg.ADAPTER_DIM).to(DEVICE)
    else: raise ValueError(method)
    
    trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
    total=sum(p.numel() for p in {id(p):p for p in model.parameters()}.values())
    print(f"[{method}] trainable {trainable/1e6:.2f}M / {total/1e6:.1f}M ({100*trainable/total:.2f}%)")
    return model

## 8. Multi-label loss, metrics, threshold calibration and submission writer
*Fixing AMP BCE overflow bugs by separating FP32 casting.*

In [ ]:
def make_pos_weight(docs):
    counts=np.zeros(N_LABELS,dtype=np.float64)
    for d in docs.values():
        for s in d["labels"]:
            if s in SID2COL: counts[SID2COL[s]]+=1
    n=len(docs)
    pw=(n-counts)/np.maximum(counts,1)
    pw=np.clip(pw,1.0,cfg.POS_WEIGHT_CAP)
    return torch.tensor(pw,dtype=torch.float32,device=DEVICE)

POS_WEIGHT=make_pos_weight(train_docs_used)

def make_optimizer(model,method):
    head_ids={id(p) for p in list(model.head.parameters())+list(model.chunk_attn.parameters())}
    body_lr=cfg.LR_BACKBONE if method=="full" else cfg.LR_PEFT
    groups=[
        {"params":[p for p in model.parameters() if p.requires_grad and id(p) in head_ids], "lr":cfg.LR_HEAD,"weight_decay":cfg.WEIGHT_DECAY},
        {"params":[p for p in model.parameters() if p.requires_grad and id(p) not in head_ids], "lr":body_lr,"weight_decay":cfg.WEIGHT_DECAY},
    ]
    return torch.optim.AdamW([g for g in groups if g["params"]],betas=(0.9,0.95),eps=1e-8)

def train_model(model,method,epochs=1,teacher=None,max_steps=None,tag=""):
    dl=DataLoader(ds_train,batch_size=cfg.BATCH_SIZE,shuffle=True,collate_fn=collate,num_workers=2,pin_memory=True)
    total_steps=min(math.ceil(len(dl)/cfg.GRAD_ACCUM)*epochs,max_steps or cfg.MAX_STEPS_PER_EXPERIMENT)
    opt=make_optimizer(model,method)
    warm=max(1,int(cfg.WARMUP_FRAC*total_steps))
    def lr_lambda(s):
        if s<warm:return (s+1)/warm
        p=(s-warm)/max(1,total_steps-warm)
        return 0.5*(1+math.cos(math.pi*min(p,1.0)))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,lr_lambda)
    scaler=torch.amp.GradScaler("cuda",enabled=cfg.AMP and USE_AMP)
    
    loss_fn=nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)
    adal=[m for m in model.modules() if isinstance(m,AdaLoRALinear)]
    if teacher is not None: teacher.eval()
    
    model.train(); opt.zero_grad(set_to_none=True)
    t0=time.time(); running=0.; seen=0; step=0
    
    for ep in range(epochs):
        for i,b in enumerate(dl):
            ids=b["input_ids"].to(DEVICE,non_blocking=True)
            tm=b["tok_mask"].to(DEVICE,non_blocking=True)
            ow=b["owner"].to(DEVICE,non_blocking=True)
            y=b["labels"].to(DEVICE,non_blocking=True)
            
            with torch.autocast("cuda",dtype=torch.float16,enabled=cfg.AMP and USE_AMP):
                logits=model(ids,tm,ow,b["n_docs"])
                if teacher is not None:
                    with torch.no_grad(): tlog=teacher(ids,tm,ow,b["n_docs"])

            # FIX: Ensure loss computation is strictly in float32 and outside autocast block to prevent c10::Half overflow
            loss = loss_fn(logits.float(), y.float())
            
            if adal:
                loss = loss + cfg.ADALORA_ORTH * sum(m.orth_penalty() for m in adal)
            if teacher is not None:
                T = cfg.KD_T
                tp = torch.sigmoid(tlog.float() / T).clamp(1e-5, 1 - 1e-5)
                sp = torch.sigmoid(logits.float() / T).clamp(1e-5, 1 - 1e-5)
                kd = F.binary_cross_entropy(sp, tp, reduction="mean") * (T * T)
                loss = (1 - cfg.KD_ALPHA) * loss + cfg.KD_ALPHA * kd
                
            scaler.scale(loss/cfg.GRAD_ACCUM).backward()
            running+=float(loss.detach()); seen+=1
            
            if (i+1)%cfg.GRAD_ACCUM==0 or i+1==len(dl):
                scaler.unscale_(opt)
                if adal: adalora_housekeeping(model,step,total_steps)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],cfg.MAX_GRAD_NORM)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                sched.step(); step+=1
                
                if step%50==0:
                    print(f"{tag} step {step}/{total_steps} loss {running/max(seen,1):.4f} elapsed {time.time()-t0:.0f}s",flush=True)
                if step>=total_steps: break
        if step>=total_steps: break
    return model

@torch.no_grad()
def predict_probs(model,ds,batch_size=2):
    model.eval()
    dl=DataLoader(ds,batch_size=batch_size,shuffle=False,collate_fn=collate,num_workers=2)
    allp=[]; allids=[]
    for b in dl:
        ids=b["input_ids"].to(DEVICE); tm=b["tok_mask"].to(DEVICE); ow=b["owner"].to(DEVICE)
        with torch.autocast("cuda",dtype=torch.float16,enabled=cfg.AMP and USE_AMP):
            logits=model(ids,tm,ow,b["n_docs"])
        allp.append(torch.sigmoid(logits.float()).cpu().numpy())
        allids.extend(b["dids"])
    return {did:p for did,p in zip(allids,np.concatenate(allp,axis=0))}

def truth_matrix(docs):
    y=np.zeros((len(docs),N_LABELS),dtype=np.int8)
    for i,d in enumerate(docs.values()):
        for s in d["labels"]:
            if s in SID2COL:y[i,SID2COL[s]]=1
    return y

def multilabel_metrics(probs,docs,thresholds):
    ids=list(docs); P=np.stack([probs[x] for x in ids])
    Y=truth_matrix({x:docs[x] for x in ids})
    pred=(P>=thresholds[None,:]).astype(np.int8)
    return {
        "macro_f1":float(f1_score(Y,pred,average="macro",zero_division=0)),
        "micro_f1":float(f1_score(Y,pred,average="micro",zero_division=0)),
        "subset_acc":float(np.mean(np.all(Y==pred,axis=1))),
    }

def tune_thresholds(probs,docs):
    ids=list(docs); P=np.stack([probs[x] for x in ids]); Y=truth_matrix({x:docs[x] for x in ids})
    global_grid=np.linspace(0.10,0.80,15)
    global_scores=[f1_score(Y,(P>=t).astype(int),average="macro",zero_division=0) for t in global_grid]
    global_t=float(global_grid[int(np.argmax(global_scores))])
    th=np.full(N_LABELS,global_t,dtype=np.float32)
    for j in range(N_LABELS):
        positives=int(Y[:,j].sum())
        if positives<3: continue
        grid=np.linspace(0.05,0.90,18)
        fs=[f1_score(Y[:,j],(P[:,j]>=t).astype(int),zero_division=0) for t in grid]
        th[j]=float(grid[int(np.argmax(fs))])
    return th,global_t

def write_submission(probs,name,thresholds):
    payload=[]
    for did in test_docs:
        p=probs[did]
        cols=np.flatnonzero(p>=thresholds)
        if len(cols)==0: cols=np.array([int(np.argmax(p))])
        pred=sorted({int(STATUTE_IDS[j]) for j in cols})
        payload.append({"id":did,"pred":pred})
        
    jp=f"{cfg.OUT_DIR}/prediction_{name}.json"
    zp=f"{cfg.OUT_DIR}/submission_{name}.zip"
    with open(jp,"w",encoding="utf-8") as f: json.dump(payload,f,ensure_ascii=False)
    with zipfile.ZipFile(zp,"w",zipfile.ZIP_DEFLATED) as z: z.write(jp,arcname="prediction.json")
    print(f"Wrote {zp} | docs {len(payload)} | avg labels {round(np.mean([len(r['pred']) for r in payload]),2)}")
    return zp

## 9. Smoke test

In [ ]:
_m=build_lsi("A")
_m=apply_adaptation(_m,"lorma")
_m=train_model(_m,"lorma",epochs=1,max_steps=5,tag="smoke")
del _m; gc.collect(); torch.cuda.empty_cache()

## 10. Run the four required architecture/technique experiments

In [ ]:
RESULTS=[]
SCORES={}
def run_experiment(name,arch,method,epochs=cfg.EPOCHS,keep_layers=None,teacher=None,write=True):
    print(f"\n========================================\n{name} | arch {arch} | technique {method}\n========================================")
    t0=time.time()
    model=build_lsi(arch,keep_layers=keep_layers)
    model=apply_adaptation(model,method)
    model=train_model(model,method,epochs=epochs,teacher=teacher,tag=name)
    
    devp=predict_probs(model,ds_dev)
    thresholds,global_t=tune_thresholds(devp,dev_docs)
    metrics=multilabel_metrics(devp,dev_docs,thresholds)
    print("Dev:",metrics,"| global threshold",global_t)
    
    testp=predict_probs(model,ds_test) if write else None
    zp=write_submission(testp,name,thresholds) if write else None
    
    if write: SCORES[name]={"dev":devp,"test":testp,"thresholds":thresholds}
    row={"name":name,"architecture":arch,"technique":method,**metrics,"global_threshold":global_t,"minutes":round((time.time()-t0)/60,1),"zip":zp}
    RESULTS.append(row)
    del model; gc.collect(); torch.cuda.empty_cache()
    return row

teacher_model = build_lsi("A")
teacher_model = apply_adaptation(teacher_model, "lorma")
teacher_model = train_model(teacher_model, "lorma", epochs=cfg.EPOCHS, tag="01_A_LoRMA (Teacher)")

# Save teacher output for ensemble
devp = predict_probs(teacher_model, ds_dev)
thresholds, global_t = tune_thresholds(devp, dev_docs)
testp = predict_probs(teacher_model, ds_test)
zp = write_submission(testp, "01_A_LoRMA", thresholds)
SCORES["01_A_LoRMA"] = {"dev": devp, "test": testp, "thresholds": thresholds}
RESULTS.append({"name": "01_A_LoRMA", "architecture": "A", "technique": "lorma", **multilabel_metrics(devp, dev_docs, thresholds), "global_threshold": global_t, "zip": zp})

run_experiment("02_B_FullFT", "B", "full")
run_experiment("03_C_AdaLoRA", "C", "adalora")
run_experiment("04_D_Adapter", "D", "adapter")

## 11. Mandatory knowledge distillation (A $\to$ 6-layer A)

In [ ]:
for p in teacher_model.parameters(): p.requires_grad = False
teacher_model.eval()

run_experiment("05_KD_A6_student", "A", "lorma", epochs=cfg.EPOCHS, keep_layers=6, teacher=teacher_model)

del teacher_model; gc.collect(); torch.cuda.empty_cache()

## 12. Ensemble and final outputs

In [ ]:
import pandas as pd
results_df = pd.DataFrame(RESULTS).sort_values("macro_f1", ascending=False)
display(results_df)
results_df.to_csv(f"{cfg.OUT_DIR}/phase5_results.csv", index=False)

available = [k for k,v in SCORES.items() if v.get("test") is not None]
if len(available) >= 2:
    ens_dev = {did: np.mean([SCORES[k]["dev"][did] for k in available], axis=0) for did in dev_docs}
    ens_test = {did: np.mean([SCORES[k]["test"][did] for k in available], axis=0) for did in test_docs}
    ens_thr, ens_global = tune_thresholds(ens_dev, dev_docs)
    print("Ensemble dev:", multilabel_metrics(ens_dev, dev_docs, ens_thr))
    write_submission(ens_test, "06_ensemble", ens_thr)